# Measurement & Statistics

Understand shot-based measurement, probability distributions, and statistical accuracy.

**Objectives:**
- See how shot count affects the precision of probability estimates
- Compute expectation values from measurement counts
- Understand the standard error and confidence in quantum measurements
- Use the `lib/utils/results` module for result processing

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.
<!-- browser-runnable -->


In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt

from lib.utils.results import parse_counts, expectation_from_counts

device = LocalSimulator()

## 1. Shot Count and Statistical Accuracy

On real quantum hardware, each circuit execution (shot) costs money.
More shots give better probability estimates but cost more.
The standard error scales as 1/sqrt(shots).

In [ ]:
# Ry(pi/3)|0> has P(|1>) = sin^2(pi/6) = 0.25 (theoretical)
theta = np.pi / 3
circuit = Circuit().ry(0, theta)
p_theory = np.sin(theta / 2) ** 2
print(f"Theoretical P(|1>) = {p_theory:.4f}")

# Run with increasing shot counts and observe convergence
shot_counts = [10, 25, 50, 100, 250, 500, 1000, 2500, 5000, 10000]
measured_p1 = []

for shots in shot_counts:
    result = device.run(circuit, shots=shots).result()
    counts = result.measurement_counts
    p1 = counts.get('1', 0) / shots
    measured_p1.append(p1)

# Plot convergence
plt.figure(figsize=(9, 4))
plt.semilogx(shot_counts, measured_p1, 'o-', label='Measured P(|1>)')
plt.axhline(y=p_theory, color='r', linestyle='--', label=f'Theory = {p_theory:.4f}')

# Show +/- 1 standard error band
std_errors = [np.sqrt(p_theory * (1 - p_theory) / n) for n in shot_counts]
plt.fill_between(shot_counts,
                 [p_theory - se for se in std_errors],
                 [p_theory + se for se in std_errors],
                 alpha=0.2, color='red', label='1 std error')

plt.xlabel('Number of shots')
plt.ylabel('P(|1>)')
plt.title('Convergence of Probability Estimate with Shot Count')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nAt 100 shots:  std error = {np.sqrt(p_theory*(1-p_theory)/100):.4f}")
print(f"At 10000 shots: std error = {np.sqrt(p_theory*(1-p_theory)/10000):.4f}")
print("10x more shots -> ~3.16x less error (sqrt scaling)")

## 2. Multi-Outcome Distributions

With multiple qubits, there are 2^n possible outcomes. Let's examine how shot count
affects our ability to resolve the probability distribution.

In [ ]:
# QFT on |000> produces a uniform distribution over 8 outcomes
from lib.circuits import qft_circuit

circuit = qft_circuit(n_qubits=3)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
shot_levels = [50, 500, 5000]

for ax, shots in zip(axes, shot_levels):
    result = device.run(circuit, shots=shots).result()
    counts = result.measurement_counts
    
    # Plot all 8 possible outcomes
    all_states = [format(i, '03b') for i in range(8)]
    probs = [counts.get(s, 0) / shots for s in all_states]
    
    ax.bar(all_states, probs, color='#232f3e')
    ax.axhline(y=1/8, color='r', linestyle='--', alpha=0.7, label='Uniform (1/8)')
    ax.set_ylim(0, 0.3)
    ax.set_title(f'{shots} shots')
    ax.set_ylabel('Measured probability')
    ax.legend(fontsize=8)

plt.suptitle('QFT|000> Should Be Uniform -- Effect of Shot Count', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Computing Expectation Values

Many quantum algorithms return their answer as an expectation value of some observable.
We can compute these from measurement counts using the `expectation_from_counts` utility.

In [ ]:
# Z-observable expectation: <Z> = P(|0>) - P(|1>)
# For Ry(theta)|0>: <Z> = cos(theta)

def z_observable(bitstring):
    """Z eigenvalue: +1 for |0>, -1 for |1>."""
    return 1.0 if bitstring == '0' else -1.0

thetas = np.linspace(0, np.pi, 15)
measured_z = []

for theta in thetas:
    circuit = Circuit().ry(0, theta)
    result = device.run(circuit, shots=5000).result()
    counts = parse_counts(result)
    exp_z = expectation_from_counts(counts, z_observable)
    measured_z.append(exp_z)

# Theory: <Z> = cos(theta)
theory_z = np.cos(thetas)

plt.figure(figsize=(8, 4))
plt.plot(thetas, measured_z, 'o', label='Measured <Z>', markersize=6)
plt.plot(thetas, theory_z, '-', label='Theory: cos(theta)', color='orange')
plt.xlabel('theta (radians)')
plt.ylabel('<Z>')
plt.title('Z-Expectation Value of Ry(theta)|0>')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from lib.circuits import bell_pair

# Multi-qubit observable: parity (XOR of all bits)
# For a Bell pair, parity is always 0 (bits always agree)

def parity_observable(bitstring):
    """Return +1 for even parity, -1 for odd parity."""
    parity = sum(int(b) for b in bitstring) % 2
    return 1.0 if parity == 0 else -1.0

# Bell pair: always even parity (00 or 11)
circuit = bell_pair()
result = device.run(circuit, shots=5000).result()
counts = parse_counts(result)
exp_parity = expectation_from_counts(counts, parity_observable)
print(f"Bell pair <parity>: {exp_parity:.4f} (expect +1.0, always even)")

# Random 2-qubit state: mixed parity
circuit_random = Circuit().h(0).h(1)
result = device.run(circuit_random, shots=5000).result()
counts = parse_counts(result)
exp_parity_random = expectation_from_counts(counts, parity_observable)
print(f"H|0>xH|0> <parity>: {exp_parity_random:.4f} (expect 0.0, random parity)")

## 4. Repeated Trials and Variance

Running the same experiment multiple times shows the variance in our estimates.

In [ ]:
# Run 50 independent trials of H|0> with 100 shots each
# Plot the distribution of measured P(|0>) values
circuit = Circuit().h(0)
n_trials = 50
shots = 100

p0_values = []
for _ in range(n_trials):
    result = device.run(circuit, shots=shots).result()
    counts = result.measurement_counts
    p0_values.append(counts.get('0', 0) / shots)

plt.figure(figsize=(8, 4))
plt.hist(p0_values, bins=15, color='#232f3e', edgecolor='white', alpha=0.8)
plt.axvline(x=0.5, color='r', linestyle='--', label='True P(|0>) = 0.5')
plt.xlabel('Measured P(|0>)')
plt.ylabel('Count')
plt.title(f'Distribution of P(|0>) over {n_trials} trials ({shots} shots each)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Mean: {np.mean(p0_values):.4f}")
print(f"Std:  {np.std(p0_values):.4f}")
print(f"Theoretical std: {np.sqrt(0.25 / shots):.4f}")

## 5. Exercises

Two exercises to make the statistics quantitative. Each has tiered hints --
expand only what you need -- and a check cell that tells you when you have it.

### Exercise 1 — Shots for one-percent precision

Section 1 showed probability estimates tightening as shots grow. Turn that
around: how many shots buy a target precision? Find the minimum shot count
that estimates P(|1>) = 0.25 with less than 1% absolute error at 95%
confidence — by calculation, not by trial and error.

Define `min_shots`: the minimum whole number of shots that meets the target.

<details><summary>Hint 1 — nudge</summary>

The standard error of a probability estimate is $\sqrt{p(1-p)/\text{shots}}$,
and a 95% confidence interval spans about 1.96 standard errors on each side.
The target caps that half-width at 0.01.

</details>
<details><summary>Hint 2 — approach</summary>

Require $1.96\sqrt{p(1-p)/\text{shots}} < 0.01$, square both sides, and
solve the inequality for shots with p = 0.25. Round up to a whole shot with
`np.ceil` (or `math.ceil`).

</details>

In [ ]:
# Exercise 1: Minimum shots to estimate P(|1>) = 0.25 within 1% at 95% confidence.
# Define: min_shots -- the minimum whole number of shots meeting the target.

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    assert min_shots > 0 and float(min_shots) == int(min_shots), (
        "shots come in whole positive numbers -- round up"
    )
    assert 1.96 * np.sqrt(0.25 * 0.75 / min_shots) <= 0.01 + 1e-9, (
        "at this shot count the 95% half-width still exceeds the 1% target"
    )
    assert 1.96 * np.sqrt(0.25 * 0.75 / (0.98 * min_shots)) > 0.01, (
        "this overshoots -- a noticeably smaller shot count already meets the target"
    )

### Exercise 2 — Measure <Z> for Ry(pi/3)|0>

Theory says the Z-expectation of Ry(pi/3)|0> is cos(pi/3) = 0.5. Verify it
with a measurement.

Define `pi3_counts`: measurement counts for the Ry(pi/3) circuit from at
least 1000 shots, and `z_pi3`: the Z-expectation value computed from those
counts.

<details><summary>Hint 1 — nudge</summary>

An expectation value is a weighted average: each bitstring contributes its
eigenvalue (+1 for |0>, -1 for |1>) weighted by how often it appeared.
Section 3 built exactly this machinery, and its observable function is still
defined.

</details>
<details><summary>Hint 2 — approach</summary>

Build a one-qubit circuit with `.ry(0, ...)` at theta = pi/3, run it on
`device` with your shot count, convert the result with `parse_counts(...)`,
then hand the counts and `z_observable` to `expectation_from_counts(...)`.

</details>

In [ ]:
# Exercise 2: Measure the Z-expectation of Ry(pi/3)|0> and compare to theory.
# Define: pi3_counts -- counts from at least 1000 shots; z_pi3 -- the
# Z-expectation computed from those counts.

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    assert sum(pi3_counts.values()) >= 1000, "run at least 1000 shots"
    assert all(len(k) == 1 for k in pi3_counts), "Ry acts on a single qubit here"
    assert pi3_counts.get("0", 0) > pi3_counts.get("1", 0), (
        "Ry(pi/3)|0> should leave |0> the more likely outcome"
    )
    assert -1.0 <= z_pi3 <= 1.0, "a Z-expectation always lives in [-1, 1]"
    assert abs(z_pi3 - 0.5) < 0.2, (
        "with 1000+ shots, <Z> should land within 0.2 of the theoretical value"
    )

## Summary

- Quantum measurement is inherently probabilistic; statistics improve with more shots
- Standard error scales as 1/sqrt(shots) -- 100x more shots for 10x better precision
- Expectation values map bitstrings to numbers and average over all shots
- On real hardware, each shot costs money -- balance precision against budget
- The `lib/utils/results` module provides `parse_counts`, `top_results`, and `expectation_from_counts`

**Next:** [05-circuit-composition.ipynb](05-circuit-composition.ipynb) -- build larger circuits from reusable components.